# Traffic Comparison between adset 43 and 43a
Comparison regarding
- feature statistics

- Prediction probabilities

Campaign: 🟠🟢 DACH // TOFU // ABO Sales // CREATIVE Testing // UGC + Static // Broad // 16.05.2025
- #043a // T: Broad (excl. 545D Purchaser Klayvio + P180D Metapixel + Inkeepr Exclusions)  // UGC Evergreen VMS Testing  // 27.01.2026

- #043 // T: Broad (excl. 545D Purchaser Klayvio + P180D Metapixel)  // UGC Evergreen VMS Testing  // 27.01.2026

In [1]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [24]:
import logging
from scipy import stats
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import awswrangler as wr
from datetime import datetime, timedelta

from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.constants import return_api_url
from general_functions.call_api_with_account_id import call_api_with_accountId, send_to_innkeepr_api_paginated

In [5]:
path_save = "SprintStories/EN-3090-traffic-comparison/data/"
customer = "ESN"
audience = "68dd4512120f694f539c0faa"
campaign = "🟠🟢 DACH // TOFU // ABO Sales // CREATIVE Testing // UGC + Static // Broad // 16.05.2025"
adsets = [
    "#043a // T: Broad (excl. 545D Purchaser Klayvio + P180D Metapixel + Inkeepr Exclusions)  // UGC Evergreen VMS Testing  // 27.01.2026",
    "#043 // T: Broad (excl. 545D Purchaser Klayvio + P180D Metapixel)  // UGC Evergreen VMS Testing  // 27.01.2026"
    ]
url = return_api_url()
url = "https://api.innkeepr.ai/api"
print(f"url = {url}")
workspace_id = return_workspace_ids()
workspace_id = [acc["id"] for acc in workspace_id if acc["name"] == customer]
workspace_id = workspace_id[0]

# Load Data

In [6]:
treatments = send_to_innkeepr_api_paginated(
    f"{url}/treatments/query",
    workspace_id,
    {"campaign": [campaign]},
    logging,
)
treatments = [t for t in treatments if t["name"] in adsets]
treatments

In [7]:
views_all = pd.read_parquet("DataChecks/views/ESN_2026-02-04.parquet")
print(f"Loaded views: {views_all.shape}")
views = views_all[views_all["treatment"].isin([t["id"] for t in treatments])]
print(f"Filtered views: {views.shape}")
views["treatment"].value_counts(dropna=False)

In [8]:
probabilities = pd.read_parquet("DataChecks/conversion_probabilities/ESN_conversion_probabilities_2026-02-04.parquet")
print(f"Loaded probabilities: {probabilities.shape}")
probabilities = probabilities[probabilities["sessionId"].isin(views["session"].unique())]
print(f"Filtered probabilities: {probabilities.shape}")
probabilities.head()

In [9]:
trial = 0
while True:
    date = datetime.today()- timedelta(days=trial)
    date = date.strftime("%Y%m%d")
    print(date)
    try:
        history = wr.s3.read_parquet(f"s3://{workspace_id}/targeting.history/{date}/{audience}.parquet")[["conv_prob","session","audience","session.date"]]
        break
    except wr.exceptions.NoFilesFound:
        print(f"No files found for {date}")
        trial += 1
    if trial == 5:
        break
history.head()

In [10]:
history = pd.merge(history, views_all[["session","treatment"]], how="left", left_on="session", right_on="session")
history["conv_prob"] = history["conv_prob"].astype(float)
history[["treatment"]].value_counts()

## merge data

In [11]:
views = pd.merge(views, probabilities[["sessionId", "probability"]], how="left", left_on="session", right_on="sessionId")
del probabilities 
views["treatment"].value_counts(dropna=False)

In [12]:
views["treatment_name"] = None
for i, treat in enumerate(treatments):
    print(treat)
    views["treatment_name"] = np.where(
        views["treatment"] == treat["id"], treat["name"], views["treatment_name"]
    )
views["label_name"] = views["treatment_name"].str.split("//").str[0]
views["treatment_name"].value_counts(dropna=False)

In [13]:
history["treatment_name"] = None
history["treatment_name"] = np.where(history["treatment"].isnull(), "direct", "some_treatment")
for i, treat in enumerate(treatments):
    print(treat)
    history["treatment_name"] = np.where(
        history["treatment"] == treat["id"], treat["name"], history["treatment_name"]
    )
history["label_name"] = history["treatment_name"].str.split("//").str[0]
history["treatment_name"].value_counts(dropna=False)

# Feature Statistics

In [14]:
views.columns

In [34]:
features = ['landingpage_cat', 'session_duration_in_s', 
       'pages_visited_per_user_cumulated', 'pages_visited_per_user',
       'pages_visited_per_session_cumulated', 'days_between_sessions',
       'pages_visited_per_session', 'session_browser_cat', 'conv_name']
# Separate the two groups
group_043 = views[views["treatment_name"].str.contains("#043 //")]
group_043a = views[views["treatment_name"].str.contains("#043a //")]
results = []

for feat in features:
    print(f"##### feature {feat} #####")
    print(views.groupby("label_name")[feat].describe())
    fig = plt.figure(figsize=(5,5))
    ax = fig.add_subplot(1, 1, 1)
    if "cat" in feat or feat == "conv_name":
        feat = feat.split("_cat")[0]
        vc = views.groupby("label_name")[feat].value_counts().reset_index()
        sns.barplot(data=vc, x=feat, y="count", hue="label_name", ax=ax)
        temp_dict = {'Significant (p<0.05)': None}
    else:
       sns.boxplot(data=views, x="label_name", y=feat, ax=ax)
       vals_043 = group_043[feat].dropna()
       vals_043a = group_043a[feat].dropna()
       if len(vals_043) > 0 and len(vals_043a) > 0:
            # Perform Mann-Whitney U test (non-parametric, doesn't assume normality)
            stat_mw, p_mw = stats.mannwhitneyu(vals_043, vals_043a, alternative='two-sided')
            
            # Perform t-test (parametric)
            stat_t, p_t = stats.ttest_ind(vals_043, vals_043a, equal_var=False)
            
            # Calculate effect size (Cohen's d)
            pooled_std = np.sqrt((vals_043.std()**2 + vals_043a.std()**2) / 2)
            cohens_d = (vals_043.mean() - vals_043a.mean()) / pooled_std if pooled_std > 0 else 0
            
            temp_dict={
                'Feature': feat,
                'Mean #043': vals_043.mean(),
                'Mean #043a': vals_043a.mean(),
                'Diff (043 - 043a)': vals_043.mean() - vals_043a.mean(),
                "Cohen's d": cohens_d,
                'T-test p-value': p_t,
                'Mann-Whitney p-value': p_mw,
                'Significant (p<0.05)': '✓' if p_t < 0.05 else 'x'
            }
            results.append(temp_dict)

    ax.set_title(f"{feat}: sign {temp_dict['Significant (p<0.05)']}")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.grid(True)
    plt.show()
    fig.savefig(f"{path_save}feat_{feat}_boxplot.png")

In [32]:
results

# Probabilities

In [16]:
views.groupby("label_name")["probability"].describe()

# History

In [36]:
history["audience"] = np.where(history["audience"].isnull(),"control",history["audience"])
hist_stats = history.groupby(by=["label_name","audience"])["conv_prob"].describe().reset_index()
hist_stats

In [23]:
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot(1, 1, 1)
sns.boxplot(data=history, x="audience", y="conv_prob", hue="label_name", ax=ax)
ax.set_title("conv_prob")
plt.xticks(rotation=90)
plt.tight_layout()
plt.grid(True)
plt.show()
fig.savefig(f"{path_save}conv_prob_boxplot.png")


In [37]:
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot(1, 1, 1)
sns.barplot(data=hist_stats, x="audience", y="count", hue="label_name", ax=ax)
ax.set_title("conv_prob")
plt.xticks(rotation=90)
plt.tight_layout()
plt.grid(True)
plt.show()
fig.savefig(f"{path_save}count_barplot.png")
